In [ ]:
# =============================================================================
# STAGE 1 - DATA PIPELINE  (unified corpus + all evaluation splits)
#
# Builds the unified requirements corpus from PROMISE_exp and SecReq, and emits
# EVERY split the three research questions need.
#
# WHAT CHANGED vs. the previous version, and why:
#
#   1. NFR sub-type labels are built here (previously a separate 01b notebook).
#      One Stage 1, one unified.parquet, one splits.json. No divergence.
#
#   2. THREE NEW SPLIT FAMILIES were added. These are the reference arm of RQ2.
#      RQ2 asks how much performance DEGRADES out of distribution. A degradation
#      is a difference, so it needs an in-distribution number to subtract from.
#      The old splits.json had only the out-of-distribution half, which made the
#      generalisation gap literally uncomputable. The new families are:
#         in_domain_fr_nfr      - stratified 5-fold CV inside PROMISE (FR/NFR)
#         in_domain_security    - stratified 5-fold CV inside PROMISE, and
#                                 separately inside SecReq (security)
#         in_domain_subtype     - stratified 5-fold CV over PROMISE NFRs, for
#                                 the all-11 / top-6 / top-4 label sets
#      With these, for every model class:
#         gap = macro_F1(in_domain) - macro_F1(cross_project | cross_dataset)
#
#   2b. RQ2 IS NOW ANSWERED FOR EVERY TASK THE DATA SUPPORTS, not just FR/NFR.
#      leave_project_out_splits() had "fr_nfr" written into it as a literal and
#      cross_dataset_splits() had the promise<->secreq pair written into it the
#      same way. Of the fifteen (task x regime) cells the Literature Review
#      promises, only seven could be filled, and the NFR sub-type task - a third
#      of the study - had no out-of-distribution arm of ANY kind. The plan is
#      now declared as data (CROSS_PROJECT_PLAN / CROSS_DATASET_TASKS) and the
#      splits are generated from it.
#
#      What that fills, and what it honestly cannot:
#          fr_nfr        in_domain  cross_project           (cross_dataset: no)
#          security      in_domain  cross_project  cross_dataset
#          subtype x3    in_domain  cross_project           (cross_dataset: no)
#
#      The two remaining gaps are DATA limits, not code limits: SecReq carries
#      neither FR/NFR nor sub-type annotation, so there is no second corpus to
#      transfer those tasks to. Closing them needs a third labelled corpus, and
#      the run logs each skipped task by name so the gap is reported, not hidden.
#
#   3. Every split entry now carries `eval_regime` (in_domain / cross_project /
#      cross_dataset) and `task`, so downstream stages never have to infer the
#      experimental condition from a fold name string.
#
#   4. A provenance manifest (stage1_manifest.json) records source URLs, raw
#      counts, de-duplication counts and final counts. The thesis quotes SecReq
#      as 510 requirements; after de-duplication this pipeline uses 444. That
#      discrepancy is now documented in a machine-readable artefact instead of
#      living only in a log line.
#
# OUTPUTS -> /kaggle/working/data_processed/
#      unified.parquet / unified.csv
#      splits.json
#      stage1_manifest.json
#
# KAGGLE: Internet = ON. Accelerator = None (this stage is CPU only).
#
# RUNNING IT ANYWHERE ELSE. Two environment variables, both unset on Kaggle so
# the defaults above are what Kaggle sees:
#      STAGE1_OUT_DIR       where the artefacts are written
#      STAGE1_LOCAL_MIRROR  a directory of pre-fetched raw files, named by their
#                           URL basename (PROMISE_exp.arff, ePurse.csv, CPN.csv,
#                           GPS.csv). Needed on any box that cannot reach
#                           gist.githubusercontent.com, which several egress
#                           allow-lists omit. The gist is also a git repository,
#                           so the mirror can be built with
#                             git clone https://gist.github.com/<gist-id>.git
#                           The manifest records the mirror path and the sha256
#                           of every raw blob consumed, so a mirrored run is
#                           never mistaken for a fetch from source.
# =============================================================================

import hashlib
import io
import json
import logging
import os
import platform
import re
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import (LeaveOneGroupOut, StratifiedGroupKFold,
                                     StratifiedKFold)

SEED = 42
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
log = logging.getLogger("stage1")

CONFIG = {
    "promise_url": ("https://raw.githubusercontent.com/AleksandarMitrevski/"
                    "se-requirements-classification/master/0-datasets/"
                    "PROMISE_exp/PROMISE_exp.arff"),
    "secreq_urls": {
        "CEPS": "https://gist.githubusercontent.com/iambackend/"
                "e8c68469c79204872ce475a64f663973/raw/ePurse.csv",
        "CPN":  "https://gist.githubusercontent.com/iambackend/"
                "e8c68469c79204872ce475a64f663973/raw/CPN.csv",
        "GPS":  "https://gist.githubusercontent.com/iambackend/"
                "e8c68469c79204872ce475a64f663973/raw/GPS.csv",
    },
    "out_dir": os.environ.get("STAGE1_OUT_DIR",
                             "/kaggle/working/data_processed"),

    # PINNED SOURCE DIGESTS. Requirement ids are positional - promise_00042 is
    # "the 43rd row the loader emitted" - and every prediction store in Stages
    # 2, 3 and 3b keys on them, as does Stage 4's join back to the corpus for
    # the per-project table. So if a source file changes upstream, ids re-point
    # to different requirements and every stored prediction silently describes
    # the wrong sentence. That is not hypothetical: the three SecReq URLs are
    # gist raw URLs with no revision in them, gists are mutable, and GPS.csv
    # currently carries a row labelled with the literal string "xyz" - exactly
    # the sort of typo an owner fixes one day.
    #
    # Recording the digests in the manifest, as this stage already did, does not
    # help: nothing ever read them back. These are the values observed on the
    # run the committed stores were built from, and a mismatch now stops the
    # run instead of corrupting it. If a change upstream is genuine and wanted,
    # update the digest here AND re-run Stages 2, 3 and 3b - the old stores
    # cannot be carried across an id change.
    "expected_sha256": {
        "PROMISE_exp.arff": "7475c2904648912ef08bd1b6149f505f7ce6ab26ee9b187c2ddee28d1af97d75",
        "ePurse.csv": "00bfbf880a163416522cac894d432f999b66efc05a62b3c1f9cc4ba3fc8fb8b3",
        "CPN.csv": "901a0278d969c578165ac82a27026a11a23f12a89cd84de40e1cf9956fc65300",
        "GPS.csv": "49fe396606ca12cbcac81326af8b55ec8d33b7351d2a82ee4ac68c6b03147ac7",
    },
    "lowercase": False,          # cased models need the original casing
    "n_folds_in_domain": 5,      # stratified CV folds for the in-domain arms
}

UNIFIED_COLUMNS = ["id", "text", "label_fr_nfr", "label_nfr_subtype",
                   "label_security", "project", "source_dataset"]

# PROMISE_exp class codes. "F" is functional; the other eleven are NFR
# sub-classes from Cleland-Huang et al. (2007). "SE" doubles as the positive
# class for the binary security task.
PROMISE_SUBTYPE_MAP = {
    "A":  "availability",
    "FT": "fault_tolerance",
    "L":  "legal",
    "LF": "look_and_feel",
    "MN": "maintainability",
    "O":  "operational",
    "PE": "performance",
    "PO": "portability",
    "SC": "scalability",
    "SE": "security",
    "US": "usability",
}
PROMISE_SECURITY_CODE = "SE"

SUBTYPE_ALL = sorted(PROMISE_SUBTYPE_MAP.values())
SUBTYPE_TOP4 = ["security", "usability", "operational", "performance"]
SUBTYPE_TOP6 = SUBTYPE_TOP4 + ["look_and_feel", "availability"]
SUBTYPE_LABELSETS = {"all": SUBTYPE_ALL, "top6": SUBTYPE_TOP6, "top4": SUBTYPE_TOP4}

# =============================================================================
# THE EVALUATION PLAN, declared as data rather than written into the split
# functions.
#
# Why this exists: leave_project_out_splits() used to have "fr_nfr" and
# "label_fr_nfr" written into it as literals, and cross_dataset_splits() had the
# promise<->secreq pair and the security task written into it the same way. The
# consequence was invisible from the code but fatal to the thesis's headline
# claim: of the fifteen (task x regime) cells the Literature Review promises,
# only seven could ever be filled. The NFR sub-type task - a third of the study -
# had NO out-of-distribution arm of any kind, so RQ2 was unanswerable for it.
#
# Declaring the plan here means adding a corpus or a task is a line in a list.
#
# Each cross-project entry is (source_dataset, task, label_col, labelset, prefix).
# Each cross-dataset task is (task, label_col, labelset); the pairs are
# enumerated from whichever corpora actually carry that label.
# =============================================================================
CROSS_PROJECT_PLAN = [
    ("promise", "fr_nfr",   "label_fr_nfr",   None, "xproj_frnfr_promise"),
    ("promise", "security", "label_security", None, "xproj_security_promise"),
    # SecReq's three sources (CEPS/ePurse, CPN, GPS) are three distinct
    # specification documents, stored in `project`. Holding one out is a genuine
    # cross-document test, and it is the ONLY arm in the study where the class
    # prior shifts sharply (CEPS 67% security, CPN 21%, GPS 36%) while the label
    # DEFINITION, the annotators and the domain all stay fixed. Every other
    # transfer arm confounds those together.
    ("secreq",  "security", "label_security", None, "xproj_security_secreq"),
] + [("promise", f"subtype_{lvl}", "label_nfr_subtype", ls,
      f"xproj_subtype_{lvl}") for lvl, ls in SUBTYPE_LABELSETS.items()]

CROSS_DATASET_TASKS = [
    ("fr_nfr",   "label_fr_nfr",   None),
    ("security", "label_security", None),
] + [(f"subtype_{lvl}", "label_nfr_subtype", ls)
     for lvl, ls in SUBTYPE_LABELSETS.items()]

# Fold names must be unique across the WHOLE splits file, because Stage 2 keys
# its resume logic on (model_tag, fold). The security pair predates this
# generalisation, so it keeps its historic names (promise_to_secreq,
# secreq_to_promise) and an existing prediction store stays valid; any other
# task carries its task name in the fold id.
LEGACY_UNSUFFIXED_TASK = "security"


# =============================================================================
# download + cleaning
# =============================================================================
# A local mirror lets Stage 1 run where the source hosts are unreachable: an
# offline machine, or a sandbox whose egress policy blocks one of them (the
# SecReq gist is on gist.githubusercontent.com, which several allow-lists do not
# carry). Point STAGE1_LOCAL_MIRROR at a directory holding the raw files under
# their URL basenames - PROMISE_exp.arff, ePurse.csv, CPN.csv, GPS.csv. The URLs
# above stay the manifest's record of provenance either way, and the manifest
# also records that a mirror was used, so a mirrored run is never mistaken for a
# fetch from source.
LOCAL_MIRROR = os.environ.get("STAGE1_LOCAL_MIRROR")

# sha256 of every raw blob this run consumed, keyed by URL, written into the
# manifest. A mirrored run and a from-source run are then comparable byte for
# byte, and a source that changes upstream is detectable instead of silent.
SOURCE_DIGESTS = {}


def _record(url, text):
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    SOURCE_DIGESTS[url] = digest
    name = url.rsplit("/", 1)[-1]
    expected = CONFIG.get("expected_sha256", {}).get(name)
    if expected and digest != expected:
        raise RuntimeError(
            f"{name} has changed upstream.\n"
            f"  expected sha256 {expected}\n"
            f"  got             {digest}\n"
            f"Requirement ids are positional, so a changed source re-points "
            f"every id and silently invalidates every stored prediction in "
            f"Stages 2, 3 and 3b. Refusing to build a corpus that no longer "
            f"matches those stores. If the change is wanted, update "
            f"CONFIG['expected_sha256'] and re-run Stages 2, 3 and 3b from "
            f"scratch - the existing stores cannot be carried across it.")
    return text


def download_text(url, retries=3, timeout=30):
    if LOCAL_MIRROR:
        cached = Path(LOCAL_MIRROR) / url.rsplit("/", 1)[-1]
        if cached.is_file():
            log.info("local mirror hit: %s <- %s", url, cached)
            return _record(url, cached.read_text(encoding="utf-8",
                                                 errors="replace"))
        log.warning("local mirror has no %s; falling back to the network.",
                    cached.name)

    last = None
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return _record(url, r.read().decode("utf-8", errors="replace"))
        except Exception as e:
            last = e
            log.warning("download attempt %d/%d failed: %s", attempt, retries, e)
            time.sleep(2 * attempt)
    raise RuntimeError(f"Could not download {url}. On Kaggle set Internet = ON. "
                       f"Last error: {last}")


_WS = re.compile(r"\s+")
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def normalize_text(s, lowercase=False):
    """Deliberately minimal. No stemming, no stop-word removal, no punctuation
    stripping: transformers use sub-word tokenizers and LLMs need natural
    sentences. Over-cleaning would silently favour one model family."""
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = _CTRL.sub(" ", s)
    s = _WS.sub(" ", s).strip()
    return s.lower() if lowercase else s


# =============================================================================
# corpus loaders
# =============================================================================
_ARFF_ROW = re.compile(r"^\s*(\d+)\s*,(.*),\s*([A-Za-z]+)\s*$")


def load_promise(cfg, manifest):
    raw = download_text(cfg["promise_url"])
    rows, unparseable = [], 0
    in_data = False
    for line in raw.splitlines():
        if not in_data:
            if line.strip().upper() == "@DATA":
                in_data = True
            continue
        if not line.strip():
            continue
        m = _ARFF_ROW.match(line)
        if not m:
            unparseable += 1
            log.warning("PROMISE: skipped unparseable line: %.80s", line)
            continue
        proj, text, code = m.group(1), m.group(2).strip(), m.group(3)
        if len(text) >= 2 and text[0] == "'" and text[-1] == "'":
            text = text[1:-1]
        text = text.replace("''", "'")
        rows.append((proj, text, code.strip().upper()))

    df = pd.DataFrame(rows, columns=["ProjectID", "RequirementText", "code"])

    # A code this loader does not recognise used to become "NFR" by default -
    # np.where(code in {F,FR,FUNCTIONAL}, "FR", "NFR") has no third branch -
    # and its sub-type became NaN. That is the corpus INVENTING a label, which
    # is exactly what load_secreq refuses to do for SecReq. GPS.csv already
    # carries a row labelled with the literal string "xyz", so the case is not
    # hypothetical. Unknown codes are now named, counted and dropped.
    _known = set(PROMISE_SUBTYPE_MAP) | {"F", "FR", "FUNCTIONAL"}
    _unknown = df[~df["code"].isin(_known)]
    n_unknown = int(len(_unknown))
    if n_unknown:
        log.warning("PROMISE: %d row(s) carry an unrecognised class code %s - "
                    "dropped rather than defaulted to NFR.",
                    n_unknown, sorted(_unknown["code"].unique()))
        df = df[df["code"].isin(_known)].reset_index(drop=True)

    out = pd.DataFrame({
        "text":              df["RequirementText"].map(lambda s: normalize_text(s, cfg["lowercase"])),
        "label_fr_nfr":      np.where(df["code"].isin(["F", "FR", "FUNCTIONAL"]), "FR", "NFR"),
        "label_nfr_subtype": df["code"].map(PROMISE_SUBTYPE_MAP),
        "label_security":    np.where(df["code"] == PROMISE_SECURITY_CODE,
                                      "security", "non-security"),
        "project":           df["ProjectID"].astype(str),
        "source_dataset":    "promise",
    })
    # Encoding damage carried by the source itself. PROMISE_exp.arff stores
    # some apostrophes as the literal three characters \92 (a cp1252 byte that
    # was never decoded). Those texts go into every prompt and every tokenizer
    # exactly as they are. They are REPORTED, not rewritten: repairing them
    # would change the text the committed predictions were produced from, and
    # the ids are positional. Quote this count in the threats to validity.
    _mojibake = int(out["text"].str.contains(r"\\9[0-9]", regex=True).sum())
    if _mojibake:
        log.warning("PROMISE: %d text(s) contain undecoded cp1252 escapes "
                    "(e.g. \\92 for an apostrophe). Left as-is so the corpus "
                    "still matches the committed predictions; reported here.",
                    _mojibake)

    manifest["promise"] = {
        "url": cfg["promise_url"],
        "raw_rows": len(out),
        "unparseable_lines_dropped": unparseable,
        "unrecognised_class_codes_dropped": n_unknown,
        "texts_with_undecoded_cp1252_escapes": _mojibake,
        "expected_rows_per_literature": 969,
        "projects": int(out["project"].nunique()),
        "fr_nfr": {k: int(v) for k, v in out["label_fr_nfr"].value_counts().items()},
    }
    log.info("PROMISE parsed: %d rows, %d projects (%d unparseable dropped).",
             len(out), out["project"].nunique(), unparseable)
    if len(out) != 969:
        log.warning("  expected 969 PROMISE_exp rows, got %d.", len(out))
    return out


_SECREQ_SEC = {"sec", "security", "1", "true", "yes"}
_SECREQ_NONSEC = {"nonsec", "non-sec", "nonsecurity", "non-security",
                  "0", "false", "no"}


def load_secreq(cfg, manifest):
    rows, per_source, skipped_total, nosep_total = [], {}, 0, 0
    for source, url in cfg["secreq_urls"].items():
        blob = download_text(url)
        n = skipped = nosep = 0
        for line in blob.splitlines():
            line = line.strip()
            if not line:
                continue
            if ";" not in line:
                # A non-empty line with no separator carries no label. It was
                # dropped silently, so a malformed export could shrink the
                # corpus with nothing in the log to say so. Counted and
                # reported, like the unrecognised-label case below.
                nosep += 1
                log.warning("SecReq[%s]: dropped line with no ';' separator: %.70s",
                            source, line)
                continue
            text, raw_label = line.rsplit(";", 1)
            text = normalize_text(text, cfg["lowercase"])
            lab = raw_label.strip().lower()
            if not text:
                continue
            if lab in _SECREQ_SEC:
                rows.append((text, "security", source)); n += 1
            elif lab in _SECREQ_NONSEC:
                rows.append((text, "non-security", source)); n += 1
            else:
                # A benchmark must never invent a label for a test point.
                skipped += 1
                log.warning("SecReq[%s]: dropped unrecognised label %r on: %.70s",
                            source, raw_label, text)
        per_source[source] = {"kept": n, "skipped_bad_label": skipped,
                              "skipped_no_separator": nosep}
        skipped_total += skipped
        nosep_total += nosep
        log.info("SecReq[%s]: %d rows (dropped %d bad-label, %d no-separator).",
                 source, n, skipped, nosep)

    out = pd.DataFrame(rows, columns=["text", "label_security", "project"])
    out["label_fr_nfr"] = np.nan       # SecReq carries no FR/NFR annotation
    out["label_nfr_subtype"] = np.nan
    out["source_dataset"] = "secreq"
    manifest["secreq"] = {
        "urls": cfg["secreq_urls"],
        "raw_rows": len(out),
        "expected_rows_per_literature": 510,
        "malformed_labels_dropped": skipped_total,
        "lines_without_separator_dropped": nosep_total,
        "per_source": per_source,
        "security": {k: int(v) for k, v in out["label_security"].value_counts().items()},
    }
    log.info("SecReq parsed: %d rows. security: %s",
             len(out), out["label_security"].value_counts().to_dict())
    return out


# =============================================================================
# unified corpus
# =============================================================================
def build_unified(promise_df, secreq_df, manifest):
    frames = []
    for src, df in [("promise", promise_df), ("secreq", secreq_df)]:
        d = df.copy().reset_index(drop=True)
        d["id"] = [f"{src}_{i:05d}" for i in range(len(d))]
        frames.append(d)
    uni = pd.concat(frames, ignore_index=True)

    before = len(uni)
    uni = uni[uni["text"].str.len() > 0]

    dup_mask = uni["text"].duplicated(keep=False)
    n_removed = int(uni["text"].duplicated().sum())
    n_cross = int((uni[dup_mask].groupby("text")["source_dataset"].nunique() > 1).sum())
    uni = uni.drop_duplicates(subset=["text"]).reset_index(drop=True)
    uni = uni[UNIFIED_COLUMNS]

    by_source = {k: int(v) for k, v in uni["source_dataset"].value_counts().items()}

    # POST-de-duplication class balance, per corpus. The raw counts recorded by
    # the loaders are the PRE-de-duplication figures quoted in the literature
    # (SecReq 187/323, PROMISE 444/525). Chapter 3 needs the counts the models
    # actually saw: they drive the class-imbalance discussion, the majority
    # baseline, and the macro- versus weighted-F1 comparison. Recording them
    # here removes any need to recompute them by hand for the write-up.
    post = {}
    for src, g in uni.groupby("source_dataset"):
        post[src] = {
            "n": int(len(g)),
            "label_security": {k: int(v) for k, v in
                               g["label_security"].value_counts(dropna=True).items()},
            "label_fr_nfr": {k: int(v) for k, v in
                             g["label_fr_nfr"].value_counts(dropna=True).items()},
            "label_nfr_subtype": {k: int(v) for k, v in
                                  g["label_nfr_subtype"].value_counts(dropna=True).items()},
        }

    manifest["deduplication"] = {
        "rows_before": before,
        "rows_after": len(uni),
        "duplicate_texts_removed": n_removed,
        "texts_appearing_in_BOTH_corpora": n_cross,   # 0 == no RQ2 leakage
        "final_by_source": by_source,
        "final_class_balance": post,
        "note": ("The literature quotes SecReq as 510 requirements and "
                 "PROMISE_exp as 969. Exact-duplicate removal reduces these to "
                 "the counts above. This must be stated in the thesis so the "
                 "experimental n reconciles with Chapter 2. 'final_class_balance' "
                 "holds the POST-de-duplication counts actually used in the "
                 "experiments; quote those, not the literature figures."),
    }
    log.info("Unified: %d rows (from %d). Removed %d duplicate texts; "
             "%d appear in BOTH corpora (cross-dataset leakage).",
             len(uni), before, n_removed, n_cross)
    if n_cross > 0:
        log.error("CROSS-CORPUS LEAKAGE DETECTED (%d texts). RQ2 is invalid "
                  "until this is resolved.", n_cross)
    log.info("By source: %s", by_source)
    log.info("NFR sub-types: %s",
             uni["label_nfr_subtype"].value_counts(dropna=False).to_dict())
    return uni


# =============================================================================
# splits
#
# Every entry is a dict with a fixed shape:
#   fold          unique name
#   family        which split family it belongs to
#   eval_regime   in_domain | cross_project | cross_dataset
#   task          fr_nfr | security | subtype_all | subtype_top6 | subtype_top4
#   label_col     column holding the gold label
#   labelset      explicit class list (multi-class only), else None
#   train_ids / test_ids
# =============================================================================
def _entry(fold, family, regime, task, label_col, tr, te, labelset=None):
    assert set(tr["id"]).isdisjoint(set(te["id"])), f"{fold}: train/test overlap"
    return {"fold": fold, "family": family, "eval_regime": regime, "task": task,
            "label_col": label_col, "labelset": labelset,
            "train_ids": tr["id"].tolist(), "test_ids": te["id"].tolist()}


def _labelled(uni, source, label_col, labelset=None):
    """Rows of one corpus that carry a usable value for one label column."""
    f = uni[(uni["source_dataset"] == source) & uni[label_col].notna()].copy()
    if labelset is not None:
        f = f[f[label_col].astype(str).isin(labelset)]
    return f.reset_index(drop=True)


def leave_project_out_splits(uni, source="promise", task="fr_nfr",
                             label_col="label_fr_nfr", labelset=None):
    """RQ2, cross-project arm at MAXIMUM granularity: one project held out per
    fold.

    Kept because 47 per-project points are what make a variance statement
    possible. "The encoder loses 5 macro-F1 on the median unseen project and 31
    on the worst" is the actual content of a cross-project claim; a single
    pooled number hides exactly the thing NoRBERT warned about.

    These folds are NOT individually scoreable and must not be reported per
    fold. On PROMISE, 24 of the 47 test sets contain only one of the two FR/NFR
    classes and two contain a single requirement, so a per-fold macro-F1 would
    score the absent class 0 and still divide by 2 - a systematic downward bias
    that is an artefact of the fold size, not of the model. Score this family
    POOLED (every requirement is tested exactly once, by a model that never saw
    its project) and take per-fold numbers from grouped_cv_splits below.
    """
    p = _labelled(uni, source, label_col, labelset)
    if p["project"].nunique() < 2:
        log.warning("LOPO %s/%s: fewer than 2 projects; skipped.", source, task)
        return
    logo = LeaveOneGroupOut()
    for tr_i, te_i in logo.split(np.arange(len(p)), groups=p["project"].values):
        tr, te = p.iloc[tr_i], p.iloc[te_i]
        held = te["project"].iloc[0]
        # NOT "cross_project". This family and grouped_cv_splits below are two
        # partitions of the SAME experiment over the SAME requirements, and
        # everything downstream groups on (task, corpus, eval_regime) without
        # reading `family`. Sharing the regime string makes them one cell in
        # which every item is scored twice. The regime is where the distinction
        # has to live, because that is the key the consumers actually read.
        yield _entry(f"LOPO_holdout_{held}", "leave_project_out",
                     "cross_project_lopo", task, label_col, tr, te, labelset)


def grouped_cv_splits(uni, source, n_folds, task, label_col, fold_prefix,
                      labelset=None):
    """RQ2, cross-project arm at REPORTABLE granularity.

    StratifiedGroupKFold holds each fold's class proportions close to the corpus
    while guaranteeing that no project appears on both sides of a split. That
    second property is the entire point: a fold sharing a project between train
    and test is not a cross-project test at all - it is in-domain evaluation
    wearing a different name, and it would inflate the number that RQ2 subtracts.

    Unlike leave-one-project-out, folds here are large enough to be
    individually meaningful. On the current corpora they hold 26-258 items.
    Every fold of the BINARY families carries both classes; three of the
    fifteen sub-type folds lack one ultra-rare class (portability and
    fault_tolerance in xproj_subtype_all_fold3, legal in fold4, availability
    in xproj_subtype_top6_fold5 - a class with ~12 corpus-wide members cannot
    appear in all five folds of a project-disjoint split). Downstream scoring
    therefore POOLS each family - every requirement is tested exactly once -
    and per-fold numbers serve only to describe spread; a sub-type fold
    missing a class must not be quoted as a standalone macro-F1.
    """
    f = _labelled(uni, source, label_col, labelset)
    if not len(f):
        log.warning("%s: no labelled rows; skipped.", fold_prefix)
        return
    groups = f["project"].astype(str).values
    y = f[label_col].astype(str).values
    n_groups = len(set(groups))
    k = min(n_folds, n_groups)
    if k < 2:
        log.warning("%s: only %d project(s); cross-project CV needs 2+. Skipped.",
                    fold_prefix, n_groups)
        return
    if k < n_folds:
        log.warning("%s: only %d project(s); reducing to %d folds.",
                    fold_prefix, n_groups, k)
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED)
    for i, (tr_i, te_i) in enumerate(sgkf.split(np.zeros(len(f)), y, groups), 1):
        tr, te = f.iloc[tr_i], f.iloc[te_i]
        # Belt and braces: the split is the claim, so assert it here as well as
        # in self_test. A silent leak here would inflate every RQ2 number.
        assert set(tr["project"]).isdisjoint(set(te["project"])), (
            f"{fold_prefix}_fold{i}: a project appears on both sides")
        yield _entry(f"{fold_prefix}_fold{i}", f"cross_project_{task}_{source}",
                     "cross_project", task, label_col, tr, te, labelset)


def cross_dataset_splits(uni, tasks=None):
    """RQ2, cross-dataset arm: every ORDERED pair of corpora that share a task.

    This used to be pinned to promise->secreq / secreq->promise on `security`,
    with a docstring asserting that security is the only shared task. That is
    true of the CURRENT two corpora, but writing it into the function meant a
    third corpus would have produced no new folds at all - the code would have
    accepted the data and silently ignored it.

    A task qualifies for a pair only when BOTH corpora carry at least two
    distinct classes for it. That is precisely why the NFR sub-type task has no
    cross-dataset arm today: PROMISE is the only corpus in the study that
    carries sub-type labels, so there is nothing to transfer to. The run logs
    each skipped task by name, so the absence is a reported result rather than a
    silent gap.

    A caveat this function cannot enforce, and which belongs in the write-up:
    two corpora sharing a COLUMN NAME do not necessarily share a label
    DEFINITION. PROMISE's `label_security` is derived from the SE sub-type, so
    it is non-functional by construction; SecReq annotates any security-relevant
    sentence, functional ones included. The arm is still valid, but part of what
    it measures is definitional shift, not only distribution shift.
    """
    tasks = tasks or CROSS_DATASET_TASKS
    sources = sorted(uni["source_dataset"].astype(str).unique())
    for task, label_col, labelset in tasks:
        have = {}
        for src in sources:
            f = _labelled(uni, src, label_col, labelset)
            if len(f) and f[label_col].astype(str).nunique() >= 2:
                have[src] = f
        if len(have) < 2:
            log.info("cross_dataset: task %r skipped - %d corpus/corpora carry "
                     "it (%s). Report this, do not omit it.",
                     task, len(have), sorted(have) or "none")
            continue
        for a in have:
            for b in have:
                if a == b:
                    continue
                name = (f"{a}_to_{b}" if task == LEGACY_UNSUFFIXED_TASK
                        else f"{a}_to_{b}_{task}")
                yield _entry(name, "cross_dataset", "cross_dataset",
                             task, label_col, have[a], have[b], labelset)


def stratified_cv_splits(frame, n_folds, family, task, label_col,
                         fold_prefix, labelset=None):
    """RQ2 REFERENCE ARM. In-domain stratified k-fold inside a single corpus.
    Without this there is no in-distribution number to subtract from, and the
    generalisation gap cannot be computed for any model class."""
    f = frame[frame[label_col].notna()].copy().reset_index(drop=True)
    if labelset is not None:
        f = f[f[label_col].isin(labelset)].reset_index(drop=True)
    y = f[label_col].astype(str).values

    # StratifiedKFold only WARNS (it does not refuse) when a class has fewer
    # members than n_splits; the resulting folds would then miss that class
    # from some test sets entirely. Reducing k keeps every class in every
    # fold. A singleton class cannot be stratified at any k, so it is dropped
    # from the frame with a loud warning rather than silently mangled.
    counts = pd.Series(y).value_counts()
    singletons = counts[counts < 2]
    if len(singletons):
        log.warning("%s: classes %s have a single member and cannot be "
                    "stratified; dropping them from this family.",
                    fold_prefix, list(singletons.index))
        f = f[~f[label_col].astype(str).isin(singletons.index)].reset_index(drop=True)
        y = f[label_col].astype(str).values
        counts = pd.Series(y).value_counts()
    too_rare = counts[counts < n_folds]
    k = n_folds
    if len(too_rare):
        k = max(2, int(counts.min()))
        log.warning("%s: classes %s have < %d members; reducing to %d folds.",
                    fold_prefix, list(too_rare.index), n_folds, k)

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    for i, (tr_i, te_i) in enumerate(skf.split(np.zeros(len(f)), y), 1):
        yield _entry(f"{fold_prefix}_fold{i}", family, "in_domain",
                     task, label_col, f.iloc[tr_i], f.iloc[te_i], labelset)


def build_all_splits(uni, cfg):
    k = cfg["n_folds_in_domain"]
    promise = uni[uni["source_dataset"] == "promise"]
    secreq = uni[uni["source_dataset"] == "secreq"]
    nfr = promise[promise["label_nfr_subtype"].notna()]

    splits = {
        # ---- RQ2 out-of-distribution arms ----------------------------------
        # Fine-grained cross-project, FR/NFR only: 47 folds, POOLED scoring,
        # kept for the per-project variance plot. See the docstring for why
        # these must not be scored per fold.
        "leave_project_out": list(leave_project_out_splits(uni)),
        "cross_dataset": list(cross_dataset_splits(uni)),

        # ---- RQ2 in-distribution reference arms (NEW) ----------------------
        "in_domain_fr_nfr": list(stratified_cv_splits(
            promise, k, "in_domain_fr_nfr", "fr_nfr",
            "label_fr_nfr", "indomain_frnfr_promise")),

        # `family` matches the dict key exactly. It used to be the truncated
        # "in_domain_security" / "in_domain_subtype" for five families, so the
        # entry's own family field could not distinguish the two security
        # corpora or the three sub-type label sets - harmless downstream
        # (Stage 2 stores the dict key), but a trap for anyone reading
        # splits.json directly.
        "in_domain_security_promise": list(stratified_cv_splits(
            promise, k, "in_domain_security_promise", "security",
            "label_security", "indomain_security_promise")),

        "in_domain_security_secreq": list(stratified_cv_splits(
            secreq, k, "in_domain_security_secreq", "security",
            "label_security", "indomain_security_secreq")),
    }

    # ---- sub-type reference arms, one per granularity level ---------------
    for level, labelset in SUBTYPE_LABELSETS.items():
        splits[f"in_domain_subtype_{level}"] = list(stratified_cv_splits(
            nfr, k, f"in_domain_subtype_{level}", f"subtype_{level}",
            "label_nfr_subtype", f"indomain_subtype_{level}", labelset=labelset))

    # ---- RQ2 cross-project arms, one per (corpus, task) -------------------
    # This is what turns RQ2 from "answered for FR/NFR only" into "answered for
    # every task the data supports". Each family is project-disjoint grouped
    # CV. Binary-task folds are individually reportable; three sub-type folds
    # lack an ultra-rare class (see grouped_cv_splits), so sub-type families
    # are quoted POOLED and per-fold numbers describe spread only.
    for source, task, label_col, labelset, prefix in CROSS_PROJECT_PLAN:
        fam = f"cross_project_{task}_{source}"
        entries = list(grouped_cv_splits(uni, source, k, task, label_col,
                                         prefix, labelset=labelset))
        if entries:
            splits[fam] = entries
        else:
            log.warning("%s produced no folds; the (task, regime) cell stays "
                        "empty and must be reported as such.", fam)

    return splits


# =============================================================================
# persist + self-test
# =============================================================================
def self_test(uni, splits):
    """Fail loudly here rather than silently producing a broken benchmark."""
    problems = []
    ids = set(uni["id"])

    for family, entries in splits.items():
        if not entries:
            problems.append(f"{family}: EMPTY")
            continue
        for e in entries:
            tr, te = set(e["train_ids"]), set(e["test_ids"])
            if tr & te:
                problems.append(f"{e['fold']}: train/test overlap")
            if not (tr | te) <= ids:
                problems.append(f"{e['fold']}: unknown ids")
            if len(te) == 0:
                problems.append(f"{e['fold']}: empty test set")
            for req in ("family", "eval_regime", "task", "label_col"):
                if not e.get(req):
                    problems.append(f"{e['fold']}: missing '{req}'")

    # in-domain arms must actually be in-domain (single corpus on both sides)
    for family, entries in splits.items():
        if not family.startswith("in_domain"):
            continue
        src = uni.set_index("id")["source_dataset"]
        for e in entries:
            if src.loc[e["train_ids"]].nunique() != 1 or src.loc[e["test_ids"]].nunique() != 1:
                problems.append(f"{e['fold']}: in-domain fold spans two corpora")

    # ---- cross-project arms must actually be project-disjoint -------------
    # This is THE claim a cross-project arm makes. A single project appearing on
    # both sides turns the fold into in-domain evaluation under another name and
    # inflates every number RQ2 subtracts from, invisibly. Groups are keyed on
    # (corpus, project) so an id collision between corpora can never mask a leak.
    grp = (uni.set_index("id")["source_dataset"].astype(str) + "::" +
           uni.set_index("id")["project"].astype(str))
    for family, entries in splits.items():
        if not entries or not str(entries[0].get("eval_regime", "")
                                   ).startswith("cross_project"):
            continue
        for e in entries:
            shared = set(grp.loc[e["train_ids"]]) & set(grp.loc[e["test_ids"]])
            if shared:
                problems.append(
                    f"{e['fold']}: PROJECT LEAK - {len(shared)} project(s) on "
                    f"both sides, e.g. {sorted(shared)[:3]}")

    # ---- cross-dataset arms must actually span two corpora ----------------
    src = uni.set_index("id")["source_dataset"].astype(str)
    for family, entries in splits.items():
        if not entries or entries[0].get("eval_regime") != "cross_dataset":
            continue
        for e in entries:
            tr_s, te_s = set(src.loc[e["train_ids"]]), set(src.loc[e["test_ids"]])
            if tr_s & te_s:
                problems.append(f"{e['fold']}: train and test share corpus "
                                f"{sorted(tr_s & te_s)}")

    # every requirement must be tested exactly once per CV family.
    # NOT applied to cross_dataset: with three or more corpora, A->C and B->C
    # both legitimately test C, so repeated test ids are expected there.
    for family, entries in splits.items():
        if entries and str(entries[0].get("eval_regime", "")).startswith(
                ("in_domain", "cross_project")):
            tested = [i for e in entries for i in e["test_ids"]]
            if len(tested) != len(set(tested)):
                problems.append(f"{family}: an item is tested more than once")

    # ---- no two families may claim the same downstream cell ---------------
    # Everything above checks one family at a time, which is exactly why the
    # worst defect in this pipeline was invisible here. Stage 4 and Stage 5 both
    # group on (task, corpus, eval_regime) and neither reads `family`, so two
    # families that agree on those three keys are ONE cell downstream and every
    # item in it is scored twice: n=1936 for a 968-item corpus, and a bootstrap
    # interval ~30% too narrow because each item is resampled twice. That is
    # what leave_project_out and cross_project_fr_nfr_promise did - same task,
    # same corpus, same regime, same 968 requirements, different partitions.
    #
    # Two families may share the key ONLY if they test disjoint items (the
    # security arms do: one tests PROMISE, the other SecReq). Overlap is the
    # error, not sharing.
    claims = {}
    for family, entries in splits.items():
        for e in entries:
            corpora = tuple(sorted(set(src.loc[e["test_ids"]])))
            key = (e["task"], corpora, e["eval_regime"])
            claims.setdefault(key, {}).setdefault(family, set()).update(e["test_ids"])
    for key, fams in claims.items():
        if len(fams) < 2:
            continue
        names = sorted(fams)
        for i, a in enumerate(names):
            for b in names[i + 1:]:
                shared = fams[a] & fams[b]
                if shared:
                    problems.append(
                        f"{a} and {b} both claim {key} and share "
                        f"{len(shared)} test item(s). Downstream keys on "
                        f"(task, corpus, eval_regime) and ignores `family`, so "
                        f"these pool into one cell and every shared item is "
                        f"scored twice. Give one of them its own eval_regime.")

    # ---- declared-class coverage per fold, REPORTED not enforced ----------
    # A project-disjoint split of an ultra-rare class (portability: 12 members
    # corpus-wide) cannot put it in all five folds, and the design accepts
    # that because those families are scored POOLED. Listing the affected
    # folds here keeps the write-up honest: any fold named below must never be
    # quoted as a standalone macro-F1.
    lab = uni.set_index("id")
    for family, entries in splits.items():
        for e in entries:
            ls = e.get("labelset")
            if not ls:
                continue
            got = set(lab.loc[e["test_ids"], e["label_col"]].dropna().astype(str))
            missing = sorted(set(ls) - got)
            if missing:
                log.warning("COVERAGE: fold %s lacks declared class(es) %s in "
                            "its test set - pooled scoring only.",
                            e["fold"], missing)

    if problems:
        for p in problems:
            log.error("SELF-TEST: %s", p)
        raise AssertionError(f"Stage 1 self-test failed with {len(problems)} problem(s).")
    log.info("SELF-TEST PASSED: %d split families, %d folds total.",
             len(splits), sum(len(v) for v in splits.values()))


def save_artifacts(uni, splits, manifest, cfg):
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    try:
        uni.to_parquet(out / "unified.parquet", index=False)
    except Exception as e:
        log.warning("parquet skipped (%s); csv still written.", e)
    uni.to_csv(out / "unified.csv", index=False)

    with open(out / "splits.json", "w") as f:
        json.dump(splits, f)

    manifest["splits"] = {
        fam: {"n_folds": len(v),
              "eval_regime": v[0]["eval_regime"] if v else None,
              "task": v[0]["task"] if v else None}
        for fam, v in splits.items()
    }
    import sklearn
    manifest["environment"] = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "seed": SEED,
        "local_mirror": LOCAL_MIRROR,
        "source_sha256": dict(SOURCE_DIGESTS),
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        # RECORDED BECAUSE IT DETERMINES THE FOLDS. StratifiedGroupKFold's
        # assignment is implementation-dependent: the same seed can yield
        # different (equally valid) cross-project folds under a different
        # scikit-learn version. The committed splits.json is therefore the
        # canonical split set; regenerating it under another sklearn produces
        # folds whose NAMES match but whose CONTENTS differ, silently
        # invalidating every resumed prediction store. Stage 2 guards against
        # exactly this by comparing stored fold contents with splits.json.
        "scikit_learn": sklearn.__version__,
    }
    with open(out / "stage1_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2, default=str)

    log.info("Saved unified corpus + %d split families -> %s",
             len(splits), out)


if __name__ == "__main__":
    manifest = {}
    promise_df = load_promise(CONFIG, manifest)
    secreq_df = load_secreq(CONFIG, manifest)
    unified = build_unified(promise_df, secreq_df, manifest)
    splits = build_all_splits(unified, CONFIG)
    self_test(unified, splits)
    save_artifacts(unified, splits, manifest, CONFIG)

    print("\n" + "=" * 74)
    print("STAGE 1 COMPLETE")
    print("=" * 74)
    print(f"  unified corpus      : {len(unified)} requirements")
    print(f"  by source           : {unified['source_dataset'].value_counts().to_dict()}")
    print(f"  NFR sub-types       : {unified['label_nfr_subtype'].nunique()} classes")

    print("\n  POST-de-duplication class balance (quote THESE in Chapter 3,")
    print("  not the pre-de-duplication figures from the literature):")
    for src, d in manifest["deduplication"]["final_class_balance"].items():
        print(f"    {src} (n={d['n']})")
        for col in ("label_fr_nfr", "label_security", "label_nfr_subtype"):
            if d[col]:
                print(f"      {col:18s} {d[col]}")
    print("\n  split families:")
    for fam, v in splits.items():
        print(f"    {fam:34s} {len(v):3d} folds   "
              f"[{v[0]['eval_regime']:14s} {v[0]['task']}]")

    # ---- RQ2 COVERAGE MATRIX --------------------------------------------
    # Printed so the experiment states its own coverage instead of leaving the
    # write-up to assume it. An empty cell here is a claim the thesis must not
    # make, and the reason it is empty is a sentence the thesis must include.
    print("\n  RQ2 COVERAGE  (task x evaluation regime)")
    print("  A blank cell is a gap. Quote this table, do not infer coverage.")
    regimes = ["in_domain", "cross_project", "cross_dataset"]
    tasks = ["fr_nfr", "security", "subtype_top4", "subtype_top6", "subtype_all"]
    cover = {(t, r): 0 for t in tasks for r in regimes}
    for v in splits.values():
        for e in v:
            # cross_project_lopo is a second partition of the cross-project
            # regime, kept separate downstream so the two are never pooled.
            # For COVERAGE it is still cross-project evidence.
            reg = ("cross_project" if str(e["eval_regime"]).startswith("cross_project")
                   else e["eval_regime"])
            key = (e["task"], reg)
            if key in cover:
                cover[key] += 1
    print(f"    {'task':14s}" + "".join(f"{r:>16s}" for r in regimes))
    filled = 0
    for t in tasks:
        cells = []
        for r in regimes:
            n = cover[(t, r)]
            filled += n > 0
            cells.append(f"{n} folds" if n else "--")
        print(f"    {t:14s}" + "".join(f"{c:>16s}" for c in cells))
    print(f"    {filled} of {len(tasks) * len(regimes)} cells covered.")
    gaps = [f"{t}/{r}" for t in tasks for r in regimes if not cover[(t, r)]]
    if gaps:
        print("    GAPS: " + ", ".join(gaps))
        print("    These are DATA limits: no second corpus in this study carries")
        print("    FR/NFR or NFR sub-type labels, so there is nothing to transfer")
        print("    those tasks to. State this in the threats to validity rather")
        print("    than claiming cross-dataset coverage the study does not have.")

    print("\n  Next: Stage 2 (fine-tuned baselines) reads unified.parquet + splits.json")
    print("=" * 74)
